# Leçon 13 - Mémoire de l'agent avec les graphes de connaissances Cognee


## Configuration

Ce carnet montre comment créer un **assistant de codage** intelligent avec une mémoire persistante en utilisant les graphes de connaissances [**Cognee**](https://www.cognee.ai/) et le **Microsoft Agent Framework** (MAF).

Cognee transforme le texte non structuré en un graphe de connaissances structuré et interrogeable, soutenu par des embeddings vectoriels — offrant à votre agent une mémoire à long terme riche et consciente des relations.

### Ce que vous allez apprendre
1. **Construire des graphes de connaissances** : Transformer les profils développeurs et les bonnes pratiques en connaissances structurées et interrogeables.
2. **Intégrer Cognee avec MAF** : Utiliser les fonctions `@tool` pour permettre à un agent MAF d’interroger le graphe de connaissances de Cognee.
3. **Conversations avec contexte de session** : Maintenir le contexte à travers plusieurs questions dans une même session.
4. **Mémoire à long terme** : Conserver des connaissances importantes entre les sessions et les récupérer dans de nouvelles conversations.

### Prérequis
- Python 3.9+
- Redis en cours d’exécution localement (`docker run -d -p 6379:6379 redis`) pour la gestion des sessions
- Une clé API LLM (par ex. OpenAI) — définir `LLM_API_KEY` dans `.env`
- `CACHING=true` dans `.env` (nécessaire pour les sessions Cognee)
- Un projet Azure AI Foundry avec un modèle de chat déployé
- `AZURE_AI_PROJECT_ENDPOINT` et `AZURE_AI_MODEL_DEPLOYMENT_NAME` dans `.env`
- Azure CLI authentifié (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Types de Mémoire de l'Agent

Ce carnet explore les mêmes trois types de mémoire que dans le carnet principal de la Leçon 13, mais utilise Cognee comme backend de mémoire à long terme :

| Type de mémoire | Mécanisme | Durée de vie |
|---|---|---|
| **Travail** | `agent.create_session()` (MAF) | Conversation unique |
| **Court terme** | Cache de session Cognee (Redis) | Session unique |
| **Long terme** | Graph de connaissances Cognee + vecteurs | Permanent |

### Architecture Mémoire de Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Préparer le stockage Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Partie 1 — Construction de la base de connaissances

Nous ingérons trois types de données pour créer une base de connaissances complète pour notre assistant de codage :

1. **Profil du développeur** — expertise personnelle et parcours technique  
2. **Meilleures pratiques Python** — le Zen de Python avec des directives pratiques  
3. **Conversations historiques** — sessions de questions-réponses passées entre développeurs et assistants IA


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiser le Graphe de Connaissances

Cognee peut rendre une visualisation HTML interactive des entités et des relations qu'il a extraites.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enrichir la mémoire avec Memify

`memify()` analyse le graphe de connaissances et génère des règles intelligentes — identifiant des motifs, des meilleures pratiques et des relations entre les concepts.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — Agent MAF avec Cognee Tools

Nous créons maintenant un agent MAF qui peut interroger le graphe de connaissances de Cognee via des fonctions `@tool`. Cela permet à l’agent de tirer parti de toute la puissance de la recherche sémantique consciente du graphe tout en conservant le contexte conversationnel grâce aux sessions.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Mémoire de travail avec des sessions

L'`AgentSession` (créée via `agent.create_session()`) fournit une mémoire de travail au sein d'une session. L'agent peut se référer aux messages précédents tout en interrogeant le graphe de connaissances à long terme de Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nouvelle session — La mémoire à long terme persiste

Commencer une nouvelle session efface la mémoire de travail, mais le graphe de connaissances Cognee reste disponible. L’agent peut récupérer les mêmes connaissances à long terme dans une toute nouvelle conversation.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Résumé

Dans ce notebook, vous avez construit un assistant de codage qui combine **la mémoire de travail de MAF** (`agent.create_session()`) avec **le graphe de connaissances à long terme de Cognee**.

### Ce que vous avez appris
1. **Construction de graphe de connaissances** : Cognee ingère du texte non structuré et construit un graphe + une mémoire vectorielle.
2. **Enrichissement du graphe avec memify** : Faits dérivés et relations plus riches par-dessus votre graphe existant.
3. **Intégration MAF + Cognee** : Les fonctions `@tool` permettent à un agent MAF d’interroger naturellement le graphe de Cognee.
4. **Mémoire de travail + mémoire à long terme** : `AgentSession` (via `agent.create_session()`) fournit le contexte de la session tandis que Cognee fournit la connaissance persistante.
5. **Recherche filtrée avec NodeSets** : Cibler des sous-ensembles spécifiques du graphe de connaissances (par ex. uniquement les principes).

### Points clés à retenir
- **Cognee** transforme le texte brut en mémoire structurée et consciente des relations — plus puissante qu’un simple magasin vectoriel.
- Les fonctions **`@tool`** font le lien proprement entre les agents MAF et les systèmes de connaissances externes.
- **`AgentSession`** (via `agent.create_session()`) conserve le contexte par conversation distinct de la connaissance durable.
- Le même graphe de connaissances sert plusieurs sessions et agents.

### Applications concrètes
- **Copilotes développeurs** : Revue de code, analyse d’incidents, assistants d’architecture
- **Copilotes orientés clients** : Agents de support sur documentation produit, FAQ, notes CRM
- **Copilotes experts internes** : Assistants politiques, juridiques ou sécurité raisonnant sur des directives
- **Couches de données unifiées** : Combiner données structurées et non structurées dans un graphe interrogeable unique

### Prochaines étapes
- Expérimenter la conscience temporelle dans Cognee
- Définir une ontologie OWL pour la qualité spécifique au domaine du graphe
- Ajouter des boucles de retour utilisateur pour améliorer la récupération dans le temps
- Monter en charge vers des systèmes multi-agents partageant la même couche de mémoire Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatiques peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour les informations critiques, une traduction professionnelle réalisée par un humain est recommandée. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
